## Great, now that we discussed a little let's continue

Given that the current approach utilized by the authors lacks reproducibility, we will explore an alternative method by leveraging nf-core pipelines for data analysis.

Please explain, how we will achieve reproducibility for the course  with this approach.


- use a standardized nf-core pipeline, so the same processing and analysis steps are applied to all samples
- specify the exact pipeline version (`-r`), ensuring that the same workflow is used when the analysis is repeated
- run the pipeline in a containerized environment (e.g. Docker), which provides the same versions of all required software and dependencies
- use the same input data and pipeline parameters for each run
- nf-core/Nextflow automatically records information about the pipeline execution, parameters and software used

The analysis can be rerun with the same data, workflow version, software environment and parameters, making the computational analysis reproducible.

---

You have successfully downloaded 2 of the fastq files we will use in our study.

What is the next step if we want to first have a count table and check the quality of our fastq files? What is the pipeline called to do so?

- Pipeline: `nf-core/rnaseq`
- performs quality control of the FASTQ files
- processes and maps the RNA-seq reads
- performs gene quantification
- generates a count table
- summarizes QC results in a MultiQC report

---

Analyze the 2 files using an nf-core pipeline.

What does this pipeline do?

- performs quality control of the raw FASTQ reads
- trims adapters and low-quality bases
- aligns reads to a reference genome
- quantifies gene/transcript expression
- produces a gene-expression count matrix and QC report

Which are the main tools that will be used in the pipeline?

- **FastQC**: raw read quality control
- **Trim Galore!**: adapter and quality trimming
- **STAR**: alignment to the reference genome
- **Salmon**: transcript/gene quantification
- **SAMtools**: processing of alignment files
- **RSeQC / Qualimap**: RNA-seq quality control
- **MultiQC**: combines QC results into one report

---

As all other nf-core pipelines, the chosen pipeline takes in a samplesheet as input.

Use Python and pandas to create the samplesheet for your 2 samples. Feel free to make use of the table you created earlier today.

Choose your sample names wisely, they must be the connection of the results to the metadata. If you can't find the sample in the metadata later, the analysis was useless.

In [1]:
from pathlib import Path
import pandas as pd

# 1. Read the two selected runs
smallest_runs = pd.read_csv(
    "smallest_runs.csv",
    header=None,
    names=["Run"]
)

# 2. Read and clean the original metadata
df = pd.read_excel("conditions_runs_oxy_project.xlsx")

metadata = pd.DataFrame({
    "Run": df["Run"],
    "Condition": df["Condition: Oxy"]
        .notna()
        .map({True: "Oxy", False: "Sal"}),
    "Genotype": df["Genotype: SNI"]
        .notna()
        .map({True: "SNI", False: "Sham"})
})

# Keep metadata only for the two selected runs
samples = smallest_runs.merge(
    metadata,
    on="Run",
    how="left"
)

# 3. Create informative sample names
samples["sample"] = (
    samples["Genotype"] + "_" +
    samples["Condition"] + "_" +
    samples["Run"]
)

# 4. Locate paired-end FASTQ files
fastq_dir = Path("fetchngs_out/fastq").resolve()

def get_fastq_files(run):
    fastq_1 = list(fastq_dir.glob(f"*{run}_1.fastq.gz"))
    fastq_2 = list(fastq_dir.glob(f"*{run}_2.fastq.gz"))

    if len(fastq_1) != 1 or len(fastq_2) != 1:
        raise FileNotFoundError(
            f"Could not uniquely find paired FASTQ files for {run}"
        )

    return str(fastq_1[0]), str(fastq_2[0])

fastqs = samples["Run"].apply(get_fastq_files)

samples["fastq_1"] = fastqs.apply(lambda x: x[0])
samples["fastq_2"] = fastqs.apply(lambda x: x[1])

# 5. Add strandedness
samples["strandedness"] = "auto"

# 6. Create nf-core/rnaseq samplesheet
samplesheet = samples[
    ["sample", "fastq_1", "fastq_2", "strandedness"]
]

# 7. Save and display
samplesheet.to_csv(
    "samplesheet_rnaseq.csv",
    index=False
)

display(samplesheet)

,sample,fastq_1,fastq_2,strandedness
0,SNI_Oxy_SRR23195516,/Users/cleostuber/computational-workflows-2026...,/Users/cleostuber/computational-workflows-2026...,auto
1,Sham_Oxy_SRR23195511,/Users/cleostuber/computational-workflows-2026...,/Users/cleostuber/computational-workflows-2026...,auto


In [4]:
# post here the command you used to run nf-core/rnaseq
!nextflow run nf-core/rnaseq \
    -r 3.27.0 \
    -profile docker,arm64 \
    -c local.config \
    -params-file salmon_params.json \
    -resume


 N E X T F L O W   ~  version 26.04.6

Launching `https://github.com/nf-core/rnaseq` [furious_rutherford] revision: a1fcdddd3b [3.27.0]

WARN: Unrecognized config option 'manifest.diagram'

------------------------------------------------------
                                        ,--./,-.
        ___     __   __   __   ___     /,-._.--~'
  |\ | |__  __ /  ` /  \ |__) |__         }  {
  | \| |       \__, \__/ |  \ |___     \`-._,-`-,
                                        `._,._,'
  nf-core/rnaseq 3.27.0
------------------------------------------------------

Input/output options
  input              : samplesheet_rnaseq.csv
  outdir             : rnaseq_results_salmon

Reference genome options
  genome             : GRCm38
  fasta              : s3://ngi-igenomes/igenomes//Mus_musculus/Ensembl/GRCm38/Sequence/WholeGenomeFasta/genome.fa
  gtf                : s3://ngi-igenomes/igenomes//Mus_musculus/Ensembl/GRCm38/Annotation/Genes/genes.gtf
  gene_bed           : s3://ngi-igenomes

Explain all the parameters you set and why you set them in this way.

- `-r 3.27.0`: fixes the `nf-core/rnaseq` version to make the analysis reproducible
- `-profile docker,arm64`: runs the pipeline in Docker containers and uses the ARM64 profile required for Apple Silicon
- `-c local.config`: loads the local resource and download settings:
    - `memory: 18.GB`: limits process memory requests to what is available to Docker on the local machine
    - `cpus: 14`: limits processes to the available CPU cores
    - `maxDownloadHeapMemory = 1.GB`: limits memory used by the AWS/S3 download client
    - `maxConnections = 100`: allows multiple simultaneous S3 connections when retrieving reference files
- `-params-file salmon_params.json`: loads the pipeline parameters from a JSON file. This was also necessary for `skip_alignment`, because Nextflow interpreted the command-line Boolean incorrectly
    - `input: samplesheet_rnaseq.csv`: provides the 2 samples and their paired-end FASTQ files
    - `outdir: rnaseq_results_salmon`: stores the results in a separate directory from the failed STAR run
    - `genome: GRCm38`: uses the mouse GRCm38 reference genome
    - `pseudo_aligner: salmon`: uses Salmon for transcript-level quantification
    - `skip_alignment: true`: skips conventional genome alignment with STAR. STAR was initially attempted but was killed while loading the large GRCm38 index because the local machine did not have enough available memory
- `-resume`: reuses successfully completed tasks from the previous run instead of recomputing them

---

## Browsing the results

How did the pipeline perform?

- the initial STAR-based run completed preprocessing and reference-preparation steps, but failed during `STAR_ALIGN` with exit code 137
- the workflow was therefore changed to Salmon pseudo-alignment with genome alignment skipped
- with `-resume`, 13 previously completed tasks were reused from cache, avoiding unnecessary recomputation
- the resumed run completed 11 additional tasks successfully, with 0 failures and 0 retries
- The successful resumed run took 27 min 57 s

---

Explain the quality control steps. Are you happy with the quality and why. If not, why not.
Please give additional information on : 
- ribosomal rRNA
- Duplication
- GC content

What are the possible steps that could lead to poorer results?

- **QC steps:** FastQC checked read quality, GC content, duplication and adapter contamination; Trim Galore removed adapters/low-quality bases; MultiQC summarized all QC results
- **Overall quality:** mostly acceptable. Per-base sequence quality passed, and adapter contamination was successfully removed by trimming. However, SNI shows high duplication and Sham has a low Salmon mapping rate (40.0%), so quality is not ideal
- **rRNA:** rRNA removal was not enabled, and no rRNA percentage is reported, so rRNA contamination cannot be assessed from this run
- **Duplication:** 
    - SNI: 59.2% raw → 58.3% trimmed
    - Sham: 28.1% → 28.1%
    - High duplication is mainly a concern for SNI
- **GC content:** 
    - SNI: 48%
    - Sham: 47% raw / 46% trimmed
    - Values are similar between samples and stable after trimming
- **Possible causes of poorer results:** RNA degradation, inefficient rRNA depletion/mRNA enrichment, excessive PCR amplification, adapter contamination, poor library preparation, sequencing errors, sample contamination, or an unsuitable reference/annotation

---

Would you exclude any samples? If yes, which and why?

No samples would be excluded; with only one sample per condition, only clear technical failure would justify removal

---

What would you now do to continue the experiment? What are the scientists trying to figure out? Which packages on R or python would you use?

- **Continue the experiment:** download and process the remaining samples with the same `nf-core/rnaseq` workflow
- **Scientific question:** determine how long-term peripheral nerve injury (SNI) and oxycodone withdrawal alter gene expression in brain reward regions (NAc, mPFC and VTA), and whether the effect of oxycodone withdrawal differs between SNI and sham animals
- **Downstream analysis:** 
    - import the Salmon gene-count matrix 
    - normalize/count-QC
    - PCA/sample clustering 
    - differential-expression analysis using the study's 2x2 design: `~ SNI + oxycodone + SNI:oxycodone` → identify DEGs and enriched pathways
- **R packages:** `tximport`/`tximeta` for Salmon counts, **`DESeq2`** for differential expression, `ggplot2` for visualization, `pheatmap` for heatmaps, `ggVennDiagram` for DEG overlaps, and `RRHO` for transcriptome-wide overlap